# Italy Road Accidents with Injuries — Time Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.

In [9]:
# Load core libraries for data processing and visualization
import pandas as pd
import numpy as np
import altair as alt

In [10]:
# Configure Altair data handling for inline specs
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [11]:
# Load and combine road-accident injury CSV batches
dataframes = []

for i in range(1, 16):
    filename = f'../data/Road accidents with injuries (IT1,41_269_DF_DCIS_INCIDENTISTR1_1,1.0) ({i}).csv'
    try:
        df = pd.read_csv(filename)
        dataframes.append(df)
    except FileNotFoundError:
        continue

combined_df = pd.concat(dataframes, ignore_index=True)
combined_df.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,ACCIDENT_LOCALIZATON,Localization of the accident,INTERSECTION,Intersection (DESC),...,Y_DEADLY_ACCIDENT,Deadly accident,HO_ROAD_ACCIDENT,Road accident hour,WEEK_DAY,Week day,MONTH,Month (DESC),TIME_PERIOD,Observation
0,A,Annual,IT,Italy,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,1,1° hour,1,Sunday,1,January,2024,48
1,A,Annual,IT,Italy,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,1,1° hour,1,Sunday,2,February,2024,45
2,A,Annual,IT,Italy,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,1,1° hour,1,Sunday,3,March,2024,62
3,A,Annual,IT,Italy,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,1,1° hour,1,Sunday,4,April,2024,52
4,A,Annual,IT,Italy,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,1,1° hour,1,Sunday,5,May,2024,69


In [12]:
# Load and combine regional population reference tables
pop_df1 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0).csv')
pop_df2 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0) (1).csv')

population_df = pd.concat([pop_df1, pop_df2], ignore_index=True)
population_df = population_df.sort_values(['REF_AREA', 'TIME_PERIOD']).reset_index(drop=True)
population_df.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,SEX,Gender,AGE,Age (DESC),MARITAL_STATUS,Marital status,TIME_PERIOD,Observation,OBS_STATUS,Observation status
0,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2019,4328565,NaN,NaN
1,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2020,4311217,NaN,NaN
2,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2021,4274945,NaN,NaN
3,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2022,4256350,NaN,NaN
4,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2023,4251351,NaN,NaN


In [13]:
# Prepare population per region/year with backfill and national total
pop_filtered = population_df[
    (population_df['Gender'] == 'Total') &
    (population_df['Age (DESC)'] == 'Total') &
    (population_df['Marital status'] == 'Total')
][['Territory', 'TIME_PERIOD', 'Observation']].copy()

pop_filtered = pop_filtered.rename(columns={'Observation': 'Population'})
pop_filtered['Territory'] = pop_filtered['Territory'].str.replace("'Valle d\"'Aosta / Vallée d\"'Aoste'", "Valle d'Aosta / Vallée d'Aoste", regex=False)

pop_2019 = pop_filtered[pop_filtered['TIME_PERIOD'] == 2019].copy()

backfilled_data = []
for year in range(2010, 2019):
    year_data = pop_2019.copy()
    year_data['TIME_PERIOD'] = year
    backfilled_data.append(year_data)

backfilled_df = pd.concat(backfilled_data, ignore_index=True)

pop_filtered = pd.concat([backfilled_df, pop_filtered], ignore_index=True).sort_values(['Territory', 'TIME_PERIOD']).reset_index(drop=True)

italy_total_pop = pop_filtered[
    pop_filtered['Territory'] != 'Trentino Alto Adige / Südtirol'
].groupby('TIME_PERIOD')['Population'].sum().reset_index()
italy_total_pop['Territory'] = 'All Regions'

population_data = pd.concat([pop_filtered, italy_total_pop], ignore_index=True)
population_data.head()

,Territory,TIME_PERIOD,Population
0,Abruzzo,2010,1300645
1,Abruzzo,2011,1300645
2,Abruzzo,2012,1300645
3,Abruzzo,2013,1300645
4,Abruzzo,2014,1300645


In [14]:
# Merge accidents with population data and compute per-capita rates
accidents_for_merge = combined_df.copy()
accidents_for_merge['Territory'] = accidents_for_merge['Territory'].replace({'Italy': 'All Regions'})
accidents_for_merge['Territory'] = accidents_for_merge['Territory'].str.replace("'Valle d\"'Aosta / Vallée d\"'Aoste'", "Valle d'Aosta / Vallée d'Aoste", regex=False)

accidents_with_pop = accidents_for_merge.merge(
    population_data,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
 )

accidents_with_pop['Accidents_Per_100k'] = (accidents_with_pop['Observation'] / accidents_with_pop['Population']) * 100000
accidents_with_pop.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,ACCIDENT_LOCALIZATON,Localization of the accident,INTERSECTION,Intersection (DESC),...,HO_ROAD_ACCIDENT,Road accident hour,WEEK_DAY,Week day,MONTH,Month (DESC),TIME_PERIOD,Observation,Population,Accidents_Per_100k
0,A,Annual,IT,All Regions,ROADACC,Road accidents with injuries,9,Total,9,Total,...,1,1° hour,1,Sunday,1,January,2024,48,58971230,0.081396
1,A,Annual,IT,All Regions,ROADACC,Road accidents with injuries,9,Total,9,Total,...,1,1° hour,1,Sunday,2,February,2024,45,58971230,0.076308
2,A,Annual,IT,All Regions,ROADACC,Road accidents with injuries,9,Total,9,Total,...,1,1° hour,1,Sunday,3,March,2024,62,58971230,0.105136
3,A,Annual,IT,All Regions,ROADACC,Road accidents with injuries,9,Total,9,Total,...,1,1° hour,1,Sunday,4,April,2024,52,58971230,0.088179
4,A,Annual,IT,All Regions,ROADACC,Road accidents with injuries,9,Total,9,Total,...,1,1° hour,1,Sunday,5,May,2024,69,58971230,0.117006


In [15]:
# Aggregate accidents by month, weekday, and hour for each region and year
chart_data = accidents_with_pop[
    (accidents_with_pop['Month (DESC)'] != 'Total') &
    (accidents_with_pop['Localization of the accident'] == 'Total') &
    (accidents_with_pop['Intersection (DESC)'] == 'Total') &
    (accidents_with_pop['Road accident type'] == 'Total')
].copy()

month_data = chart_data.groupby([
    'Territory', 'TIME_PERIOD', 'Month (DESC)'
]).agg({
    'Observation': 'sum',
    'Population': 'first',
    'Accidents_Per_100k': 'sum'
}).reset_index()
month_data['View'] = 'Month'
month_data['Time_Value'] = month_data['Month (DESC)']

day_data = chart_data[chart_data['Week day'] != 'Total'].groupby([
    'Territory', 'TIME_PERIOD', 'Week day'
]).agg({
    'Observation': 'sum',
    'Population': 'first',
    'Accidents_Per_100k': 'sum'
}).reset_index()
day_data['View'] = 'Day'
day_data['Time_Value'] = day_data['Week day']

hour_data = chart_data[chart_data['Road accident hour'] != 'Total'].groupby([
    'Territory', 'TIME_PERIOD', 'Road accident hour'
]).agg({
    'Observation': 'sum',
    'Population': 'first',
    'Accidents_Per_100k': 'sum'
}).reset_index()
hour_data['View'] = 'Hour'
hour_data['Time_Value'] = hour_data['Road accident hour'].replace('Unknown hour', 'Unknown')
hour_data['Time_Value'] = hour_data['Time_Value'].str.replace('° hour', 'h', regex=False)
hour_data['Time_Value'] = hour_data['Time_Value'].replace('0h', '24h')

combined_chart_data = pd.concat([
    month_data[['Territory', 'TIME_PERIOD', 'View', 'Time_Value', 'Observation', 'Population', 'Accidents_Per_100k']],
    day_data[['Territory', 'TIME_PERIOD', 'View', 'Time_Value', 'Observation', 'Population', 'Accidents_Per_100k']],
    hour_data[['Territory', 'TIME_PERIOD', 'View', 'Time_Value', 'Observation', 'Population', 'Accidents_Per_100k']]
], ignore_index=True)

combined_chart_data.head()

,Territory,TIME_PERIOD,View,Time_Value,Observation,Population,Accidents_Per_100k
0,Abruzzo,2010,Month,April,1372,1300645,105.486124
1,Abruzzo,2010,Month,August,1628,1300645,125.168666
2,Abruzzo,2010,Month,December,1016,1300645,78.115089
3,Abruzzo,2010,Month,February,1064,1300645,81.805566
4,Abruzzo,2010,Month,January,1156,1300645,88.878979


In [16]:
# Build interactive Altair chart for accidents across month/day/hour by region
from pathlib import Path

available_regions = combined_chart_data[
    ~combined_chart_data['Territory'].isin(['Provincia Autonoma Bolzano / Bozen', 'Provincia Autonoma Trento'])
]['Territory'].unique().tolist()
available_regions = sorted([r for r in available_regions if r != 'All Regions'])

view_dropdown = alt.binding_select(options=['Month', 'Day', 'Hour'], name='Time View: ')
view_selection = alt.selection_point(fields=['View'], bind=view_dropdown, value='Month')

available_years = sorted(combined_chart_data['TIME_PERIOD'].unique())
year_dropdown = alt.binding_select(options=available_years, name='Year: ')
year_selection = alt.selection_point(fields=['TIME_PERIOD'], bind=year_dropdown, value=2024)

region_selection = alt.selection_point(fields=['Territory'], value=[{'Territory': 'All Regions'}])

chart_data_filtered = combined_chart_data[
    ~combined_chart_data['Territory'].isin(['Provincia Autonoma Bolzano / Bozen', 'Provincia Autonoma Trento'])
].copy()

month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hour_order = ['24h'] + [f'{i}h' for i in range(1, 24)]
all_time_order = month_order + day_order + hour_order + ['Unknown']

all_territories = sorted([t for t in chart_data_filtered['Territory'].unique() if t != 'All Regions'])
territory_order = ['All Regions'] + all_territories

custom_colors = ['#4169E1'] + ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']

line_chart = alt.Chart(chart_data_filtered).mark_line(point=True).encode(
    x=alt.X('Time_Value:N', title='Time Period', axis=alt.Axis(labelAngle=-45), sort=all_time_order),
    y=alt.Y('Accidents_Per_100k:Q', title='Accidents per 100,000 Inhabitants'),
    color=alt.Color('Territory:N', title='Region', scale=alt.Scale(domain=territory_order, range=custom_colors), sort=territory_order),
    opacity=alt.condition(region_selection, alt.value(1), alt.value(0.2)),
    tooltip=[
        alt.Tooltip('Territory:N', title='Region'),
        alt.Tooltip('TIME_PERIOD:O', title='Year'),
        alt.Tooltip('View:N', title='Time View'),
        alt.Tooltip('Time_Value:N', title='Time Period'),
        alt.Tooltip('Accidents_Per_100k:Q', title='Accidents per 100k', format=',.2f'),
        alt.Tooltip('Observation:Q', title='Total Accidents', format=',.0f'),
        alt.Tooltip('Population:Q', title='Population', format=',.0f')
    ]
).add_params(
    view_selection,
    year_selection,
    region_selection
).transform_filter(
    view_selection
).transform_filter(
    year_selection
).properties(
    width=800,
    height=400,
    title={
        'text': 'Road Accidents with Injuries in Italy - Time Analysis',
        'subtitle': 'Source: IstatData (https://esploradati.istat.it/databrowser/#/en). | Click on a line to isolate.',
        'anchor': 'start'
    }
)

graphs_dir = Path('../graphs')
graphs_dir.mkdir(parents=True, exist_ok=True)
chart_out = graphs_dir / 'italy_road_accidents_with_injuries_time_analysis.json'
line_chart.save(chart_out, format='json')

line_chart

alt.Chart(...)